# Nested CV — `FFT_PhaseLag_PLI`

**Sheet**: `FFT_PhaseLag_PLI`  
**Bands** (10): `Delta, Theta, Alpha, Beta, HighBeta, Alpha1, Alpha2, Beta1, Beta2, Beta3`

Raw (non-z-scored) sheet. StandardScaler inside the pipeline is fit only on training data each fold — fully leakage-free.

## Methodology (mirrors 3_* reference notebooks)
- **Outer CV**: `StratifiedGroupKFold`, 5 splits × 2 repeats = 10 folds (patient-level groups)
- **Inner CV**: 5-fold `GridSearchCV` inside every band-subset evaluation
- **Band-subset search**: all non-empty subsets of the candidate bands (exhaustive)
- **Features per band**: mean, std, global_mean, global_std, median across channels/pairs → 5 values/band
- **Model**: Logistic Regression (balanced class weights), L1/L2 × liblinear/saga × C grid
- **Threshold tuning**: threshold swept on outer-train predictions to maximise balanced accuracy
- **Metrics**: balanced accuracy, F1, sensitivity, specificity, average precision


In [ ]:
import os, re, glob, itertools
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, recall_score, average_precision_score
)

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGK = True
except Exception:
    HAS_SGK = False


# ── Sheet-specific config ────────────────────────────────────────────────────
sheet_name       = 'FFT_PhaseLag_PLI'
selected_columns = ['Delta', 'Theta', 'Alpha', 'Beta', 'HighBeta', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Beta3']

# ── Outcomes ─────────────────────────────────────────────────────────────────
df = pd.read_excel('processeddata/Randomization factors and Primary outcome.xlsx')
diff = df[df['Event Name'].isin(['T1', 'T2'])].pivot_table(
    index='Patient number', columns='Event Name', values='Pain Unpleasantness'
).dropna()
diff['Diff'] = diff['T2'] - diff['T1']

output_dir        = 'processeddata'
median_threshold  = -2.00

# ── LogReg grid ───────────────────────────────────────────────────────────────
solvers  = ['liblinear', 'saga']
penalties = ['l1', 'l2']
C_values  = [1e-3, 1e-2, 1e-1, 1, 10, 100]

min_patients_required = 20

# ── CV config ─────────────────────────────────────────────────────────────────
outer_n_splits  = 5
outer_n_repeats = 2
inner_n_splits  = 5


# ── Utilities ─────────────────────────────────────────────────────────────────
def parse_patient_id_from_filename(xlsx_path):
    m = re.search(r'CIPN3(\d{3})', os.path.basename(xlsx_path))
    return int(m.group(1)) if m else None


def build_samples(output_dir):
    files = sorted(glob.glob(os.path.join(output_dir, 'CIPN3*.xlsx')))
    rows = []
    for f in files:
        pid = parse_patient_id_from_filename(f)
        if pid is not None:
            rows.append((pid, f))
    return rows


def all_nonempty_subsets(cols):
    cols = list(cols)
    out = []
    for r in range(1, len(cols) + 1):
        out.extend(list(itertools.combinations(cols, r)))
    return out


def feats_from_single_band_column(df_band):
    """
    df_band : single-column DataFrame for one band (rows = channels / pairs).
    Returns 5 features: [col_mean, col_std, global_mean, global_std, global_median].
    """
    A = df_band.values.astype(float)
    A = np.nan_to_num(A, nan=0.0, posinf=0.0, neginf=0.0)
    col_mean = A.mean(axis=0)[0]
    col_std  = A.std(axis=0)[0]
    g_mean   = A.mean()
    g_std    = A.std()
    g_med    = float(np.median(A))
    return np.array([col_mean, col_std, g_mean, g_std, g_med], dtype=float)


def load_bandwise_cache_for_file(xlsx_path, sheet_name, bands):
    """Returns dict band -> (5,) feature vector."""
    df_sheet = pd.read_excel(xlsx_path, sheet_name=sheet_name, index_col=0)
    out = {}
    for b in bands:
        if b in df_sheet.columns:
            out[b] = feats_from_single_band_column(df_sheet[[b]])
    return out


def build_X_from_subset(cache_list, subset):
    """Build feature matrix from a list of band-cache dicts and a band subset."""
    X_rows = []
    for d in cache_list:
        if any(b not in d for b in subset):
            return None
        X_rows.append(np.concatenate([d[b] for b in subset], axis=0))
    return np.vstack(X_rows)


def make_outer_splits(X, y, groups, n_splits, n_repeats, base_seed=42):
    all_splits = []
    for r in range(n_repeats):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True,
                                   random_state=base_seed + r)
        for tr, te in cv.split(X, y, groups=groups):
            all_splits.append((tr, te))
    return all_splits


def mean_std(x):
    x = np.asarray(x, dtype=float)
    return float(x.mean()), float(x.std())


# ── Build patient sample list ─────────────────────────────────────────────────
if not HAS_SGK:
    raise RuntimeError('sklearn version does not have StratifiedGroupKFold. '
                       'Upgrade with: pip install -U scikit-learn')

rows = build_samples(output_dir)
rows = [(pid, f) for (pid, f) in rows if pid in diff.index]

if len(rows) == 0:
    raise RuntimeError('No samples found after filtering for labelled patients.')

print(f'Found {len(rows)} files with labels.')

unique_pids = sorted(set(pid for pid, _ in rows))
y_patient   = (diff.loc[unique_pids, 'Diff'] >= median_threshold).astype(int)
pid_to_y    = y_patient.to_dict()

y      = np.asarray([int(pid_to_y[pid]) for pid, _ in rows], dtype=int)
groups = np.asarray([pid for pid, _ in rows], dtype=int)
kept_files = [f for _, f in rows]

print(f'Unique patients : {len(np.unique(groups))}')
print(f'Threshold {median_threshold} | '
      f'Neg (improvers)={int((y==0).sum())} '
      f'Pos (non-impr)={int((y==1).sum())}')

if len(np.unique(groups)) < min_patients_required:
    raise RuntimeError(f'Too few patients: {len(np.unique(groups))}')


# ── Pre-load band-wise feature caches ────────────────────────────────────────
print(f"\nLoading sheet='{sheet_name}' for bands={selected_columns}")
band_cache_all = []
ok_count = 0
for f in kept_files:
    try:
        d = load_bandwise_cache_for_file(f, sheet_name, selected_columns)
        band_cache_all.append(d)
        ok_count += 1
    except Exception as e:
        band_cache_all.append({})
        print(f'  [WARN] {os.path.basename(f)}: {e}')

print(f'Loaded caches for {ok_count}/{len(kept_files)} files.')

if all(len(d) == 0 for d in band_cache_all):
    raise RuntimeError(f"No band columns found in sheet '{sheet_name}'.")


# ── Model + grid ──────────────────────────────────────────────────────────────
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced', max_iter=50000,
        tol=1e-3, random_state=42
    ))
])

param_grid = []
for solver in solvers:
    for penalty in penalties:
        entry = {'clf__solver': [solver], 'clf__penalty': [penalty],
                 'clf__C': C_values}
        if penalty == 'l1':
            entry['clf__warm_start'] = [True]
        param_grid.append(entry)


# ── Outer splits ──────────────────────────────────────────────────────────────
X_dummy      = np.zeros((len(y), 1), dtype=float)
outer_splits = make_outer_splits(X_dummy, y, groups,
                                  outer_n_splits, outer_n_repeats, base_seed=42)
print(f'\nOuter CV: {outer_n_splits} splits × {outer_n_repeats} repeats '
      f'= {len(outer_splits)} folds.')

candidate_subsets = all_nonempty_subsets(selected_columns)
print(f'Band-subset search space: {len(candidate_subsets)} subsets.')


# ── Nested CV ─────────────────────────────────────────────────────────────────
test_metrics  = {'bal_acc': [], 'f1': [], 'recall_pos': [],
                  'recall_neg': [], 'ap': []}
train_metrics = {'bal_acc': [], 'f1': [], 'recall_pos': [],
                  'recall_neg': [], 'ap': []}
chosen_subsets = []

for fold_i, (tr, te) in enumerate(outer_splits, 1):
    g_tr, g_te = groups[tr], groups[te]

    # Sanity: no patient overlap
    assert len(set(g_tr) & set(g_te)) == 0, \
        f'Fold {fold_i}: patient overlap between train and test!'

    print('\n' + '='*80)
    print(f'[Fold {fold_i}/{len(outer_splits)}]  '
          f'train={len(tr)} test={len(te)}  '
          f'train_patients={len(np.unique(g_tr))} '
          f'test_patients={len(np.unique(g_te))}')

    inner_cv = StratifiedGroupKFold(n_splits=inner_n_splits,
                                     shuffle=True, random_state=1)

    best_subset      = None
    best_inner_score = -1
    best_gs          = None
    best_X_tr        = None
    best_X_te        = None

    cache_tr = [band_cache_all[i] for i in tr]
    cache_te = [band_cache_all[i] for i in te]

    progress_every = max(1, len(candidate_subsets) // 10)

    for si, subset in enumerate(candidate_subsets, 1):
        X_tr_sub = build_X_from_subset(cache_tr, subset)
        if X_tr_sub is None:
            continue
        X_te_sub = build_X_from_subset(cache_te, subset)
        if X_te_sub is None:
            continue

        gs = GridSearchCV(
            pipe, param_grid=param_grid,
            scoring='balanced_accuracy',
            cv=inner_cv, n_jobs=-1, refit=True
        )
        gs.fit(X_tr_sub, y[tr], groups=g_tr)

        if gs.best_score_ > best_inner_score:
            best_inner_score = gs.best_score_
            best_subset      = subset
            best_gs          = gs
            best_X_tr        = X_tr_sub
            best_X_te        = X_te_sub

        if si % progress_every == 0:
            print(f'  {si}/{len(candidate_subsets)} subsets | '
                  f'best so far: {best_subset} score={best_inner_score:.3f}')

    if best_subset is None:
        raise RuntimeError('No valid subset found.')

    chosen_subsets.append(best_subset)
    print(f'\nBest subset : {best_subset}  inner_BA={best_inner_score:.3f}')
    print(f'Best params : {best_gs.best_params_}')

    best_est = best_gs.best_estimator_

    p_tr = best_est.predict_proba(best_X_tr)[:, 1]
    p_te = best_est.predict_proba(best_X_te)[:, 1]

    # Tune threshold on outer-train to maximise balanced accuracy
    ths = np.linspace(0.05, 0.95, 181)
    best_th, best_ba = 0.5, -1.0
    for t in ths:
        ba = balanced_accuracy_score(y[tr], (p_tr >= t).astype(int))
        if ba > best_ba:
            best_ba, best_th = ba, t

    yhat_tr = (p_tr >= best_th).astype(int)
    yhat_te = (p_te >= best_th).astype(int)

    for split, yhat, proba, y_true, mdict in [
        ('train', yhat_tr, p_tr, y[tr],  train_metrics),
        ('test',  yhat_te, p_te, y[te],  test_metrics),
    ]:
        mdict['bal_acc'].append(balanced_accuracy_score(y_true, yhat))
        mdict['f1'].append(f1_score(y_true, yhat, zero_division=0))
        mdict['recall_pos'].append(
            recall_score(y_true, yhat, pos_label=1, zero_division=0))
        mdict['recall_neg'].append(
            recall_score(y_true, yhat, pos_label=0, zero_division=0))
        mdict['ap'].append(average_precision_score(y_true, proba))

    print(f'Threshold tuned on train: th={best_th:.3f}  '
          f'train_BA={train_metrics["bal_acc"][-1]:.3f}  '
          f'test_BA={test_metrics["bal_acc"][-1]:.3f}  '
          f'test_AP={test_metrics["ap"][-1]:.3f}')


# ── Summary ───────────────────────────────────────────────────────────────────
tb,  ts  = mean_std(test_metrics['bal_acc'])
tap, tas = mean_std(test_metrics['ap'])
rn,  rns = mean_std(test_metrics['recall_neg'])
rp,  rps = mean_std(test_metrics['recall_pos'])
tf1, tfs = mean_std(test_metrics['f1'])
trb, _   = mean_std(train_metrics['bal_acc'])
trap, _  = mean_std(train_metrics['ap'])

print('\n' + '='*80)
print(f'[{sheet_name}] patient-wise nested CV + all-subsets band selection')
print(f'TEST  bal_acc={tb:.3f}±{ts:.3f} | AP={tap:.3f}±{tas:.3f} | '
      f'recall_neg={rn:.3f}±{rns:.3f} | recall_pos={rp:.3f}±{rps:.3f} | '
      f'F1={tf1:.3f}±{tfs:.3f}')
print(f'TRAIN bal_acc={trb:.3f} | AP={trap:.3f}')

print('\nChosen band subsets per outer fold:')
for i, ss in enumerate(chosen_subsets, 1):
    print(f'  Fold {i:>2d}: {ss}')

from collections import Counter
band_counts = Counter()
for ss in chosen_subsets:
    for b in ss:
        band_counts[b] += 1

print('\nBand selection frequency across outer folds:')
for b in selected_columns:
    print(f'  {b:<12s}: {band_counts[b]}/{len(chosen_subsets)}')
